# AI Fundamentals for Data Professionals
## Complete Masterclass: LLMs, RAG, Vector Databases & AI Agents

**Objective**: Master AI fundamentals including LLMs, embeddings, vector databases, and building RAG systems using LangChain.

**Prerequisites**: Python and SQL knowledge are required. No prior AI experience needed.

**Duration**: Comprehensive hands-on learning

---

## Table of Contents
1. Setting Up Your Development Environment
2. Understanding Large Language Models (LLMs)
3. Making API Calls to LLMs
4. Exploring Tokenization and Embeddings
5. Introduction to Vector Databases
6. Building a Retrieval-Augmented Generation (RAG) System
7. Prompt Engineering Types
8. Building AI Agents with React Architecture

## Section 1: Setting Up Your Development Environment

### Required Tools
1. **VS Code** - Code editor (download from code.visualstudio.com)
2. **Python 3.11+** - Download from python.org (avoid 3.14, use 3.11, 3.12, or 3.13)
3. **UV** - Modern Python package manager (built on Rust, faster than pip)
4. **Virtual Environment** - Isolated Python environment for the project

### API Keys Required
- **OpenAI API Key**: For ChatGPT and GPT models
- **Google Gemini API Key**: For Google's Gemini models (optional but recommended)

### Best Practices
- Never hardcode API keys in your code
- Always use `.env` files to store sensitive information
- Keep `.env.example` in version control (without real values)
- Load environment variables using `python-dotenv`

In [ ]:
# Step 1: Installation Commands
# Run these in your terminal (not in the notebook)

# 1. Install UV package manager
# pip install uv

# 2. Initialize project
# uv init

# 3. Sync dependencies
# uv sync

# 4. Add required packages
# uv add python-dotenv openai google-generativeai langchain langchain-openai langchain-community pypdf chromadb

# 5. Create .env file with your API keys
# OPENAI_API_KEY=your_key_here
# GEMINI_API_KEY=your_key_here

In [ ]:
# Import required libraries
from dotenv import load_dotenv
import os
from openai import OpenAI
import google.generativeai as genai

# Load environment variables
load_dotenv()

# Check if API keys are set
openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

print("✓ OpenAI API Key loaded" if openai_key else "✗ OpenAI API Key not found")
print("✓ Gemini API Key loaded" if gemini_key else "✗ Gemini API Key not found")

## Section 2: Understanding Large Language Models (LLMs)

### What is an LLM?

An **Large Language Model** is:
- An application that predicts the next word (or token) based on context
- Trained on massive amounts of text data
- Built using Transformer architecture (not RNN or statistical methods)

### Simple Definition
> Language Model = Application that answers "What is the next word?" given context

**Example**: 
- Input: "The cat will sit on the ___"
- Output: Probability distribution over possible next words (mat, sofa, chair, etc.)

### Evolution of Language Models
1. **Statistical Era** (1990s-2010s)
   - N-grams, Bigrams, Trigrams
   - Problem: Limited context understanding, Gmail-like approaches
   
2. **Neural Network Era** (2010s)
   - RNNs, LSTMs, GRUs
   - Problem: Hard to debug, required machine learning expertise
   
3. **Transformer Era** (2018+)
   - Transformer architecture
   - Introduced the concept of "Attention"
   - Foundation for modern LLMs (GPT, Gemini, Claude)

### Major LLM Providers
- **OpenAI**: GPT-4, GPT-4 Mini (Chat interface)
- **Google**: Gemini (Bard alternative)
- **Anthropic**: Claude
- **Meta**: Llama (Open source)
- **Other Players**: Grok, Cohere, etc.

### Key Insight
You don't need to build LLMs from scratch—big organizations have already built them. Your job is to use them effectively!

## Section 3: Making API Calls to LLMs

### Difference Between Chat GPT Website and API Calls
- **Chat GPT Website**: User-friendly interface for general Q&A
- **API Calls**: Direct programmatic access to LLM models
- As a developer, you need direct API access to build custom solutions

### Why Not Just Use Chat GPT?
- Not suitable for production applications
- Can't integrate with your organization's data
- Can't automate workflows
- Can't handle batch processing
- Not cost-effective for large-scale use

### Price Comparison (OpenAI GPT-4 Mini)
- **Input**: $0.25 per 1 million tokens
- **Output**: $2 per 1 million tokens
- **Batch Processing**: 50% cheaper (slower, non-realtime)
- **Note**: 1 token ≈ 1 word (roughly)

In [ ]:
# Example 1: OpenAI API Call
# Create OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Make API call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Tell me a fun fact about the data domain"}
    ]
)

print("=== OpenAI Response ===")
print(response.choices[0].message.content)

In [ ]:
# Example 2: Google Gemini API Call
# Configure Gemini
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-2.5-flash')

# Make API call
response = model.generate_content("Tell me a fun fact about the data domain")

print("=== Google Gemini Response ===")
print(response.text)

## Section 4: Understanding Tokenization and Embeddings

### Step 1: Tokenization
> **Definition**: Breaking down text into smaller units called "tokens"

**Key Concepts**:
- 1 token ≈ 1 word (roughly)
- Sometimes: "playing" = "play" + "ing" (2 tokens)
- Hyphens: "hands-on" = "hands" + "-" + "on" (3 tokens)
- Each token has a unique ID in the model's vocabulary

**Example**:
```
Text: "I am eating chocolate"
Tokens: ["I"] ["am"] ["eating"] ["chocolate"]
Token IDs: [40] [716] [660] [11311]
```

### Step 2: Static Embeddings
> **Definition**: Converting tokens into vectors (numbers with direction)

**Key Concepts**:
- Each token ID maps to a vector (e.g., [0.1, -0.9, 0.999, ...])
- OpenAI standard: 1,536 dimensions per vector
- Fixed meaning for each token (doesn't consider context)
- Example: "bank" always has the same embedding whether it means financial bank or river bank

### Step 3: Contextual Embeddings (Transformer Magic!)
> **Definition**: Converting tokens to vectors considering surrounding context

**How it works**:
- Each word "looks at" all other words in the sentence
- Relationships are learned (e.g., "cat" relates differently to "sat on" vs "fed")
- Final embedding depends on context, not just the token itself

**The Power**: 
- "turn on the fan" (device) vs "I am your fan" (admirer) → Different embeddings!
- This is the heart of transformer architecture

In [ ]:
# Create embeddings for a sentence
from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Create embeddings
text = "I am a data professional learning AI"
response = openai_client.embeddings.create(
    input=text,
    model="text-embedding-3-small"
)

# Extract the embedding vector
embedding_vector = response.data[0].embedding

print(f"Text: {text}")
print(f"Embedding dimension: {len(embedding_vector)}")  # Should be 1536
print(f"First 10 values: {embedding_vector[:10]}")
print(f"\nFull embedding vector length: {len(embedding_vector)}")

# This vector represents the semantic meaning of the entire sentence!
# It can be compared with other sentence embeddings using distance metrics

## Section 5: Introduction to Vector Databases

### Problem We're Trying to Solve
**Scenario**: Passing 1000+ page PDFs with every query
- Problems:
  1. User effort: Must upload PDFs every time
  2. Token cost: 1000 pages = 700k+ tokens per query
  3. Hallucination risk: Might miss relevant information

### Solution: Vector Database
> Store your documents once. Retrieve only relevant chunks per query.

### Core Concept: Chunking + Vector Storage
**Two-part data storage**:
1. **Context** = Embedding vector (semantic meaning)
2. **Content** = Actual text (for final answer generation)

**Example**:
```
Document: "Company was founded in 2016. Started with 5 people."
    ↓ Chunk
Chunk 1: "Company was founded in 2016."
    ├─ Context: Embedding vector [0.1, -0.9, ...]
    └─ Content: "Company was founded in 2016."
Chunk 2: "Started with 5 people."
    ├─ Context: Embedding vector [0.2, 0.8, ...]
    └─ Content: "Started with 5 people."
```

### Popular Vector Database Options
1. **Chroma** (Free, Open-source) - Best for learning
2. **PostgreSQL + pgvector** (Open-source) - For relational data
3. **Pinecone** (Hosted, Paid) - Production-ready
4. **Weaviate** (Open-source & Hosted) - Enterprise solution
5. **AWS opensearch** - For AWS users
6. **Azure Search** - For Azure users

## Section 6: Building RAG (Retrieval-Augmented Generation)

### What is RAG?
> **RAG = Retrieval + Augmented + Generation**

1. **Retrieval**: Fetch relevant documents from vector DB
2. **Augmented**: Combine retrieved docs with user query
3. **Generation**: Use LLM to generate answer from augmented context

### The RAG Pipeline

**Step 1: Document Processing**
```
PDF/Text → Chunking → Embedding → Vector DB
```

**Step 2: Query Processing**
```
User Query → Convert to Vector → Search Vector DB → Rank by Similarity
```

**Step 3: Answer Generation**
```
Top-K Similar Chunks + Query → LLM → Final Answer
```

### Similarity Search Algorithm

**Key Concept**: Vector Distance  
- Formula: `distance = 1 - cos(θ)` 
- Vectors close in space = Similar meaning
- Compare query vector with all stored vectors
- Rank by smallest distance

**Example**:
```
Vector DB has 1000 chunks
Query Vector searches for relevant chunks
Only returns Top-3 matching chunks
Reduces token cost from 1M+ to just the relevant parts!
```

### Cost Reduction Example
**Without RAG**:
- Query: 20 tokens
- Document: 1,000 pages = 700,000 tokens
- Total: **700,020 tokens** per query × $0.25 per million = $0.18 per query

**With RAG**:
- Query: 20 tokens
- Retrieved chunks: ~2,000 tokens (from 700,000)
- Total: **2,020 tokens** per query × $0.25 per million = $0.0005 per query
- **Savings: 99.7%** cost reduction!

In [ ]:
# Step 1: Load and chunk documents (RAG Pipeline)
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Example: Creating embeddings for text chunks
# (In real scenario, you would load a PDF file here)

# Create a sample document
sample_text = [
    {
        "content": "Novasphere Technology was founded in 2016 with a mission to provide data solutions.",
        "metadata": {"page": 0, "source": "sample.pdf"}
    },
    {
        "content": "By 2019, the company had grown to more than 15 employees.",
        "metadata": {"page": 1, "source": "sample.pdf"}
    },
    {
        "content": "The organization specializes in cloud migration and data analytics.",
        "metadata": {"page": 2, "source": "sample.pdf"}
    }
]

print("Sample documents created:")
for i, doc in enumerate(sample_text):
    print(f"{i+1}. {doc['content'][:50]}...")


In [ ]:
# Step 2: Create text splitter for chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,      # Characters per chunk
    chunk_overlap=20     # Overlap to preserve context
)

# Simulate chunking
chunks = []
for doc in sample_text:
    split_chunks = splitter.split_text(doc['content'])
    for chunk_text in split_chunks:
        chunks.append({
            'content': chunk_text,
            'metadata': doc['metadata']
        })

print(f"Total chunks created: {len(chunks)}")
print("\nChunk examples:")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk['content']}")
    print(f"Metadata: {chunk['metadata']}\n")

In [ ]:
# Step 3: Create embeddings and store in vector database
from langchain.schema import Document

# Convert chunks to LangChain Document format
documents = [
    Document(page_content=chunk['content'], metadata=chunk['metadata'])
    for chunk in chunks
]

# Initialize embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create Chroma vector database and store embeddings
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"  # Save locally
)

print(f"✓ Vector database created with {len(documents)} documents")
print(f"✓ Database persisted to './chroma_db'")
print(f"✓ Each document converted to 1,536-dimensional embedding")

In [ ]:
# Step 4: Similarity Search (Retrieval)
# Query the vector database for relevant chunks

query = "When was Novasphere Technology founded?"

# Perform similarity search
results = vector_db.similarity_search(query, k=3)  # Top 3 results

print(f"Query: {query}")
print(f"\n=== Retrieved Documents ===")
for i, result in enumerate(results):
    print(f"\n{i+1}. Content: {result.page_content}")
    print(f"   Metadata: {result.metadata}")
    
# Behind the scenes:
# 1. Query converted to 1,536-dimensional vector
# 2. Compared with all document vectors using cosine similarity
# 3. Top-3 most similar documents returned

In [ ]:
# Step 5: Generate Answer Using LLM (Generation)
from langchain_openai import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prepare context from retrieved documents
context = "\n".join([doc.page_content for doc in results])

# Create prompt with context and query
prompt = f"""Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer:"""

# Generate answer
response = llm.invoke(prompt)

print(f"Query: {query}")
print(f"\nRetrieved Context:")
print(context)
print(f"\n=== LLM Generated Answer ===")
print(response.content)

## Section 7: Prompt Engineering Types

### Why Prompt Engineering Matters
- 10% of RAG implementation but 100% of user experience
- Communicating requirements to LLM is an art and science
- Users won't write perfect prompts - you need to handle that

### Type 1: Zero-Shot Prompting
> Ask LLM to perform task **without providing examples**

**Relies on**: Pre-trained knowledge
**Best for**: General, straightforward tasks
**Pros**: Fast, no examples needed
**Cons**: May not handle complex tasks well

**Example**:
```
User: "Translate to French: How are you?"
LLM: "Comment ça va?" or "Comment vas-tu?"
```

### Type 2: One-Shot Prompting
> Provide **one example** of desired output

**Best for**: Teaching a pattern with one reference
**Helps**: Fine-tune behavior on-the-fly

**Example**:
```
Good morning → Bonjour
How are you → ?

LLM: "Comment allez-vous?" or "Ça va?"
```

### Type 3: Few-Shot Prompting
> Provide **multiple examples** for better understanding

**Best for**: Complex patterns, specific formatting
**Helps**: Better guidance through patterns

**Example**:
```
Good morning → Bonjour
Thank you → Merci
How are you → ?

LLM: "Comment allez-vous?" (more confident)
```

### Type 4: Chain-of-Thought Prompting
> Make LLM **explain step-by-step** before answering

**Key insight**: LLMs need guidance for complex reasoning

**Best for**: Mathematics, logic puzzles, decision-making
**Works because**: Mimics how humans solve complex problems

**Example**:
```
Problem: "If you have 3 apples and get 2 more, how many apples do you have?"

Chain-of-Thought:
Step 1: You start with 3 apples
Step 2: You receive 2 more apples  
Step 3: Add them together: 3 + 2 = 5
Answer: 5 apples
```

### Temperature Parameter
Controls randomness in LLM responses:
- **Temperature = 0**: Always pick highest probability token (deterministic)
- **Temperature = 0.5**: Medium randomness (balanced)
- **Temperature = 1.0**: High randomness (creative)

**Use case:**
- Q&A, Facts: Temperature = 0
- Creative writing: Temperature = 0.7-1.0

In [ ]:
# Prompting Techniques Demonstration

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("=== ZERO-SHOT PROMPTING ===")
zero_shot = llm.invoke("Translate to French: How are you?")
print(f"Response: {zero_shot.content}\n")

print("=== ONE-SHOT PROMPTING ===")
one_shot = llm.invoke("""
Example:
Good morning → Bonjour

Now translate:
How are you?
""")
print(f"Response: {one_shot.content}\n")

print("=== FEW-SHOT PROMPTING ===")
few_shot = llm.invoke("""
Examples:
Good morning → Bonjour
Thank you → Merci
Goodbye → Au revoir

Now translate:
How are you?
""")
print(f"Response: {few_shot.content}\n")

print("=== CHAIN-OF-THOUGHT PROMPTING ===")
cot = llm.invoke("""
Solve step by step:
If you have 3 apples and get 2 more, how many apples do you have?

Step 1: Start with...
Step 2: Receive...
Step 3: Calculate...
Answer:
""")
print(f"Response: {cot.content}")

## Section 8: Building AI Agents with ReAct Architecture

### What is an AI Agent?
An **AI Agent** is:
- An LLM equipped with tools
- Can reason about which tools to use
- Takes actions based on reasoning
- Iterates until task is complete

### Real-World Scenario: Manual Process
```
Manager asks Rahul: "How much data did we get today?"
Rahul thinks: "I don't know, let me check..."
Rahul's Actions:
  1. Query SQL database
  2. Get result: "1 GB of data"  
  3. Report back to manager
```

### AI Agent Version
```
User asks Agent: "How much data did we get today?"
Agent reasons: "I need to check the database"
Agent actions:
  1. Use SQL tool to query
  2. Parse response
  3. Generate answer
```

### ReAct Framework
> **ReAct = Reasoning + Acting**

**How it works**:
1. **Reason**: Analyze what information is needed
2. **Act**: Call appropriate tool
3. **Observe**: Get tool result
4. **Reason again**: Is answer complete?
5. **Act again**: Call more tools if needed
6. **Generate**: Final answer when complete

### Key Components

**1. Tools**
- SQL Database Access
- Web Search
- File Operations
- Calculator
- Custom APIs

**2. Tool Binding**
- Connect tools to LLM
- LLM knows when/how to use each tool
- Automatic tool calling

**3. Orchestration Loop**
```
while not conversation_complete:
    reason_about_task()
    decide_on_action()
    call_tool()
    observe_result()
    update_context()
```

### Example Multi-Step Task
```
User: "What's the latest AI news AND find a research paper about it?"

Agent Process:
├─ Reason: Need news + research
├─ Act: Call Google Search tool
│   └─ Get: "Latest AI news about transformers"
├─ Reason: Now need research paper
├─ Act: Call Research Paper tool  
│   └─ Get: Paper on transformer improvements
├─ Reason: Have all info needed
└─ Generate: Final comprehensive answer
```

### Why ReAct is Powerful
- ✓ Handles complex, multi-step tasks
- ✓ Adapts to unexpected situations
- ✓ Can correct itself
- ✓ Transparent reasoning process
- ✓ Extensible with new tools

In [ ]:
# Simple Agent Example with LangChain

from langchain.agents import tool
from langchain_openai import ChatOpenAI
from typing import Any

# Step 1: Define Tools
@tool
def database_query(query: str) -> str:
    """Query company database for information"""
    # In real scenario, this would query actual DB
    simulated_results = {
        "employee count": "150 employees",
        "revenue 2024": "$5 million",
        "data size": "50 TB",
        "cloud platform": "AWS"
    }
    return simulated_results.get(query.lower(), "No data found")

@tool
def web_search(query: str) -> str:
    """Search the web for information"""
    # In real scenario, this would use actual search
    return f"Web search results for: {query}"

# Step 2: Create Tool List
tools = [database_query, web_search]

print("Available Tools:")
for tool_item in tools:
    print(f"- {tool_item.name}: {tool_item.description}")

# Step 3: Create Agent
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create the agent
agent = create_tool_calling_agent(llm, tools)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Step 4: Use Agent
print("\n=== Running Agent ===")
response = executor.invoke({
    "input": "How many employees do we have and what is our current data size?"
})

print(f"\nFinal Answer: {response['output']}")

## Summary: Your AI Development Roadmap

### Core Concepts Covered ✓
1. **LLMs**: Understanding what they are and how they work
2. **Tokenization**: Breaking text into tokens with IDs
3. **Embeddings**: Converting text to semantic vectors
4. **Vector Databases**: Storing and retrieving embeddings
5. **RAG**: Building cost-efficient Q&A systems
6. **Prompting**: Different techniques for querying LLMs
7. **Agents**: Multi-step reasoning with tools

### Key Takeaways for Data Professionals
- RAG is YOUR domain (data retrieval & augmentation)
- LLMs will become standard in every organization soon (just like SQL)
- 90% of implementation = RAG, 10% = Prompt Engineering
- Start building with open-source tools (Chroma, LangChain)
- Focus on cost optimization through intelligent retrieval

### Practical Path Forward
1. **Setup Phase**: Install tools, get API keys
2. **Learning Phase**: Run examples in this notebook
3. **Building Phase**: Create RAG system for your data
4. **Optimization Phase**: Fine-tune prompts & retrieval
5. **Scaling Phase**: Move to production databases (Pinecone, etc)

### Common Misconceptions
❌ "AI is a bubble"
✓ AI is a revolution similar to computers and internet

❌ "I need to understand all of transformer math"
✓ High-level understanding is enough to build solutions

❌ "Prompt engineering is not important"
✓ It's 10% of implementation but 100% of user experience

### Resources for Further Learning
- [LangChain Documentation](https://python.langchain.com)
- [OpenAI API Reference](https://platform.openai.com/docs)
- [Chroma Vector DB](https://www.trychroma.com)
- [Semantic Search Concepts](https://en.wikipedia.org/wiki/Semantic_search)

### Your Next Steps
1. Create your own PDF files to test RAG
2. Experiment with different embedding models
3. Build an agent with custom tools for your use case
4. Explore semantic chunking for better retrieval
5. Deploy your RAG system to production

---

**Remember**: The future is not about learning AI as a specialist in a separate domain. 
It's about integrating AI into your existing role—whether you're a data analyst, engineer, or professional.

**Stay curious. Keep building. Keep learning!** 🚀

## Quick Reference Cheat Sheet

### Installation Commands
```bash
# Create project
uv init

# Add dependencies
uv add python-dotenv openai google-generativeai langchain langchain-openai langchain-community pypdf chromadb

# Activate environment (if needed)
source .venv/bin/activate  # Linux/Mac
.venv\Scripts\activate     # Windows
```

### Create API Keys
1. **OpenAI**: https://platform.openai.com/api-keys
2. **Gemini**: https://aistudio.google.com/app/apikey

### Environment Setup (.env file)
```
OPENAI_API_KEY=sk-...
GEMINI_API_KEY=AIza...
```

### Common Code Patterns

**OpenAI Chat Completion**
```python
from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Your question"}]
)
```

**Create Embeddings**
```python
from openai import OpenAI
client = OpenAI()
embedding = client.embeddings.create(
    input="Text to embed",
    model="text-embedding-3-small"
)
vector = embedding.data[0].embedding  # 1536-dimensional vector
```

**RAG Pipeline**
```python
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# Load → Split → Embed → Store
loader = PyPDFLoader("file.pdf")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
chunks = splitter.split_documents(docs)
vector_db = Chroma.from_documents(chunks, OpenAIEmbeddings())

# Query
results = vector_db.similarity_search("Your question", k=3)
```

### Pricing Quick Reference
- **GPT-4o Mini**: $0.25/$2 per million tokens (input/output)
- **Embeddings (small)**: $0.02 per million tokens
- **5 tokens** ≈ 1 word (rough estimate)

### Common Issues & Solutions
| Issue | Solution |
|-------|----------|
| API key not found | Check .env file and load_dotenv() call |
| High token count | Use RAG to retrieve only relevant chunks |
| Slow queries | Use batch processing or cached vectors |
| Inaccurate results | Improve prompt or add few-shot examples |
| High costs | Implement RAG for 99%+ cost reduction |